# 02b. IVI 재설계 — 돌봄 인프라 접근성 추가
## 사회적 고립 취약도 다차원 확장 (IVI v2)

**재설계 배경**

기존 IVI는 `elderly_ratio`(고령화율) + `elderly_alone_ratio`(독거율) 2개 변수만으로  
사회적 고립을 측정했습니다. 그러나 사회적 고립은 단순히 '혼자 사는가'가 아니라  
**'돌봄 자원에 얼마나 접근할 수 있는가'** 를 함께 봐야 합니다.

**추가 변수: 경로당 접근성 (구 단위)**

- 데이터: 서울시 경로당 정보 (3,644개소, 2025.08.13 갱신)
- 집계 단위: **구(자치구) 단위** — 아래 사유로 행정동 직접 집계 불가
  1. 주소(도로명)에서 법정동 추출 가능 비율 59.2%에 불과
  2. 법정동 → 행정동은 1:N 관계 (지오코딩 필요)
  3. 2025년 데이터로 2021년 행정동 경계와 완전 매핑 불확실
- 활용 방식: 구별 경로당 밀도 (경로당/고령인구 1000명)를 **구 단위 돌봄 인프라 보정 계수**로 적용

**시간 불일치 사유 및 처리 방침**

경로당은 의료시설 대비 입지 변동성이 낮은 구조적 사회 인프라입니다.  
2025년 현황이 2021년과 완전히 동일하지는 않으나, 구별 밀도 순위는 안정적으로 유지됩니다.  
본 분석에서는 **구조적 돌봄 인프라 격차** 파악 목적으로 활용하며, 보고서에 한계로 명시합니다.

**IVI v2 공식**

$$\text{IVI}_{v2} = \text{elderly\_ratio\_score} \times 0.35 + \text{elderly\_alone\_score} \times 0.35 + \text{care\_shortage\_score} \times 0.30$$

| 변수 | 가중치 | 의미 |
|------|--------|------|
| elderly_ratio_score | 0.35 | 지역 내 고령화 정도 |
| elderly_alone_score | 0.35 | 독거노인 비율 (사회적 단절) |
| care_shortage_score | 0.30 | 구별 경로당 부족도 (돌봄 인프라 역수) |

---
## 1. 라이브러리 및 경로 설정

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE_DIR   = Path('c:/Tsum2026/T_SUM2026')
PROC_DIR   = BASE_DIR / 'data' / 'processed'
RAW_DIR    = BASE_DIR / 'data' / 'raw' / 'IVI_social_isolation'

IVI_PATH         = PROC_DIR / 'ivi_social_isolation_2021.csv'
GYEONG_PATH      = RAW_DIR  / '서울시 경로당 정보.csv'
OUT_IVI_V2       = PROC_DIR / 'ivi_social_isolation_2021_v2.csv'

WEIGHTS = {'elderly_ratio': 0.35, 'elderly_alone': 0.35, 'care_shortage': 0.30}

for label, path in [
    ('IVI v1', IVI_PATH),
    ('경로당 정보', GYEONG_PATH),
    ('출력 폴더', PROC_DIR),
]:
    mark = '✓' if path.exists() else '✗ 없음!'
    print(f'[{mark}] {label}: {path}')

[✓] IVI v1: c:\Tsum2026\T_SUM2026\data\processed\ivi_social_isolation_2021.csv
[✓] 경로당 정보: c:\Tsum2026\T_SUM2026\data\raw\IVI_social_isolation\서울시 경로당 정보.csv
[✓] 출력 폴더: c:\Tsum2026\T_SUM2026\data\processed


---
## 2. 데이터 불러오기

In [2]:
ivi = pd.read_csv(IVI_PATH, encoding='utf-8-sig')
grd = pd.read_csv(GYEONG_PATH, encoding='cp949')

print(f'IVI v1 shape: {ivi.shape}')
print(f'경로당 shape: {grd.shape}')
print()
print('IVI 컬럼:', ivi.columns.tolist())
print('경로당 컬럼:', grd.columns.tolist())
print()
print('경로당 시설종류:', grd['시설종류'].unique().tolist())

IVI v1 shape: (426, 14)
경로당 shape: (3644, 9)

IVI 컬럼: ['year', 'gu', 'dong', 'dong_key', 'total_population', 'elderly_population', 'elderly_alone', 'elderly_ratio', 'elderly_alone_ratio', 'elderly_ratio_score', 'elderly_alone_score', 'IVI', 'IVI_rank', 'IVI_grade']
경로당 컬럼: ['연번', '시도명', '시군구명', '시설종류', '시설명(경로당명)', '주소(도로명)', '지자체명', '담당부서', '전화번호']

경로당 시설종류: ['노인여가복지시설']


---
## 3. 구별 경로당 밀도 산출

경로당 수 / (고령인구 / 1000) → 고령자 1000명당 경로당 수

In [3]:
# 구별 경로당 수
grd_by_gu = grd.groupby('시군구명').size().reset_index(name='경로당수')
grd_by_gu = grd_by_gu.rename(columns={'시군구명': 'gu'})

# IVI에서 구별 고령인구 집계
eld_by_gu = ivi.groupby('gu')['elderly_population'].sum().reset_index()
eld_by_gu.columns = ['gu', '구별_고령인구']

# 병합
care = grd_by_gu.merge(eld_by_gu, on='gu', how='outer')

# 고령자 1000명당 경로당 수
care['경로당_per_1000eld'] = care['경로당수'] / (care['구별_고령인구'] / 1000)

print('=== 구별 경로당 밀도 (고령자 1000명당) ===')
display(
    care.sort_values('경로당_per_1000eld')
    [['gu', '경로당수', '구별_고령인구', '경로당_per_1000eld']]
    .round(2)
    .reset_index(drop=True)
)
print(f'\n경로당 밀도 평균: {care["경로당_per_1000eld"].mean():.2f}')
print(f'최소 구: {care.loc[care["경로당_per_1000eld"].idxmin(), "gu"]} '
      f'({care["경로당_per_1000eld"].min():.2f})')
print(f'최대 구: {care.loc[care["경로당_per_1000eld"].idxmax(), "gu"]} '
      f'({care["경로당_per_1000eld"].max():.2f})')

=== 구별 경로당 밀도 (고령자 1000명당) ===


,gu,경로당수,구별_고령인구,경로당_per_1000eld
0,관악구,116,79871.0,1.45
1,강북구,103,64333.0,1.60
2,송파구,181,97691.0,1.85
3,중랑구,136,71682.0,1.90
4,금천구,78,41041.0,1.90
5,강동구,141,74070.0,1.90
6,광진구,99,51723.0,1.91
7,은평구,169,87241.0,1.94
8,중구,50,24392.0,2.05
9,도봉구,137,64160.0,2.14



경로당 밀도 평균: 2.29
최소 구: 관악구 (1.45)
최대 구: 성동구 (3.58)


---
## 4. 돌봄 인프라 부족 점수 산출

$$\text{care\_access\_score} = \text{MinMax}(\text{경로당\_per\_1000eld})$$
$$\text{care\_shortage\_score} = 1 - \text{care\_access\_score}$$

점수가 높을수록 경로당이 부족한 구 = 돌봄 인프라 취약

In [4]:
mn = care['경로당_per_1000eld'].min()
mx = care['경로당_per_1000eld'].max()
care['care_access_score']   = (care['경로당_per_1000eld'] - mn) / (mx - mn)
care['care_shortage_score'] = 1 - care['care_access_score']

print('=== 돌봄 인프라 부족 점수 (높을수록 취약) ===')
display(
    care.sort_values('care_shortage_score', ascending=False)
    [['gu', '경로당_per_1000eld', 'care_access_score', 'care_shortage_score']]
    .round(3)
    .reset_index(drop=True)
)

=== 돌봄 인프라 부족 점수 (높을수록 취약) ===


,gu,경로당_per_1000eld,care_access_score,care_shortage_score
0,관악구,1.452,0.000,1.000
1,강북구,1.601,0.070,0.930
2,송파구,1.853,0.188,0.812
3,중랑구,1.897,0.209,0.791
4,금천구,1.901,0.211,0.789
5,강동구,1.904,0.212,0.788
6,광진구,1.914,0.217,0.783
7,은평구,1.937,0.228,0.772
8,중구,2.050,0.281,0.719
9,도봉구,2.135,0.321,0.679


---
## 5. IVI v2 산출

$$\text{IVI}_{v2} = \text{elderly\_ratio\_score} \times 0.35 + \text{elderly\_alone\_score} \times 0.35 + \text{care\_shortage\_score} \times 0.30$$

In [5]:
# care_shortage_score 병합
df = ivi.merge(care[['gu', 'care_shortage_score', '경로당_per_1000eld']], on='gu', how='left')

miss = df['care_shortage_score'].isna().sum()
if miss > 0:
    print(f'care_shortage_score 결측: {miss}건')
    print(df[df['care_shortage_score'].isna()][['gu','dong']].to_string())
    df['care_shortage_score'].fillna(df['care_shortage_score'].mean(), inplace=True)
    print('  → 전체 평균으로 대체')

# IVI v2 = 가중 합산 (모든 성분이 [0,1]이므로 추가 MinMax 불필요)
df['IVI_v2'] = (
    df['elderly_ratio_score'] * WEIGHTS['elderly_ratio'] +
    df['elderly_alone_score'] * WEIGHTS['elderly_alone'] +
    df['care_shortage_score'] * WEIGHTS['care_shortage']
)

# 순위 및 등급
df['IVI_v2_rank']  = df['IVI_v2'].rank(ascending=False, method='min').astype(int)
df['IVI_v2_grade'] = pd.qcut(
    df['IVI_v2'].rank(method='first'), q=5,
    labels=[1, 2, 3, 4, 5], duplicates='drop'
).astype(int)

print('=== IVI v1 vs v2 분포 비교 ===')
compare_stats = pd.DataFrame({
    'IVI_v1': df['IVI'].describe(),
    'IVI_v2': df['IVI_v2'].describe()
}).round(4)
display(compare_stats)

print()
print(f'IVI v1 표준편차: {df["IVI"].std():.4f}')
print(f'IVI v2 표준편차: {df["IVI_v2"].std():.4f}')
print(f'v1-v2 상관계수: {df["IVI"].corr(df["IVI_v2"]):.4f}')

=== IVI v1 vs v2 분포 비교 ===


,IVI_v1,IVI_v2
count,426.0000,426.0000
mean,0.3865,0.4522
std,0.1155,0.1101
min,0.1040,0.1984
25%,0.3067,0.3790
50%,0.3810,0.4466
75%,0.4504,0.5247
max,0.8224,0.8460



IVI v1 표준편차: 0.1155
IVI v2 표준편차: 0.1101
v1-v2 상관계수: 0.8110


---
## 6. 순위 변동 분석

In [6]:
df['rank_change'] = df['IVI_rank'] - df['IVI_v2_rank']  # 양수 = 순위 상승 (더 취약하다고 평가)

print('=== IVI v2 상위 20개 (가장 사회적 고립 취약한 동) ===')
display(
    df.nsmallest(20, 'IVI_v2_rank')[
        ['gu', 'dong', 'elderly_ratio_score', 'elderly_alone_score',
         'care_shortage_score', 'IVI', 'IVI_v2', 'IVI_v2_rank', 'rank_change']
    ].round(3)
)

print()
print('=== 순위 가장 많이 상승한 동 (돌봄 부족 구에 속해 새롭게 부각) ===')
display(
    df.nlargest(10, 'rank_change')[
        ['gu', 'dong', 'IVI', 'IVI_v2', 'IVI_rank', 'IVI_v2_rank', 'rank_change', 'care_shortage_score']
    ].round(3)
)

print()
print('=== 순위 가장 많이 하락한 동 (경로당 풍부한 구에 속해 재평가) ===')
display(
    df.nsmallest(10, 'rank_change')[
        ['gu', 'dong', 'IVI', 'IVI_v2', 'IVI_rank', 'IVI_v2_rank', 'rank_change', 'care_shortage_score']
    ].round(3)
)

=== IVI v2 상위 20개 (가장 사회적 고립 취약한 동) ===


,gu,dong,elderly_ratio_score,elderly_alone_score,care_shortage_score,IVI,IVI_v2,IVI_v2_rank,rank_change
137,강북구,번3동,0.846,0.774,0.930,0.810,0.846,1,2
23,중구,을지로동,0.719,0.913,0.719,0.816,0.787,2,0
136,강북구,번2동,0.739,0.707,0.930,0.723,0.785,3,4
8,종로구,종로1.2.3.4가동,0.640,0.975,0.601,0.808,0.746,4,0
133,강북구,송천동,0.650,0.678,0.930,0.664,0.744,5,6
379,강남구,수서동,1.000,0.645,0.559,0.822,0.743,6,-5
252,강서구,가양2동,0.917,0.643,0.560,0.780,0.714,7,-2
337,관악구,삼성동,0.741,0.430,1.000,0.586,0.710,8,12
18,중구,회현동,0.746,0.662,0.719,0.704,0.708,9,-1
138,강북구,수유1동,0.574,0.640,0.930,0.607,0.704,10,5



=== 순위 가장 많이 상승한 동 (돌봄 부족 구에 속해 새롭게 부각) ===


,gu,dong,IVI,IVI_v2,IVI_rank,IVI_v2_rank,rank_change,care_shortage_score
333,관악구,신림동,0.264,0.485,372,158,214,1.0
323,관악구,낙성대동,0.286,0.500,353,140,213,1.0
336,관악구,대학동,0.308,0.515,318,120,198,1.0
329,관악구,서원동,0.326,0.528,296,99,197,1.0
324,관악구,청룡동,0.315,0.521,309,115,194,1.0
328,관악구,남현동,0.321,0.525,301,108,193,1.0
331,관악구,서림동,0.335,0.535,286,93,193,1.0
327,관악구,인헌동,0.339,0.537,279,89,190,1.0
322,관악구,행운동,0.349,0.544,264,81,183,1.0
332,관악구,신사동,0.358,0.550,248,75,173,1.0



=== 순위 가장 많이 하락한 동 (경로당 풍부한 구에 속해 재평가) ===


,gu,dong,IVI,IVI_v2,IVI_rank,IVI_v2_rank,rank_change,care_shortage_score
64,성동구,용답동,0.459,0.321,96,376,-280,0.000
61,성동구,성수2가1동,0.435,0.305,123,387,-264,0.000
63,성동구,송정동,0.420,0.294,142,394,-252,0.000
50,성동구,마장동,0.402,0.281,179,402,-223,0.000
55,성동구,금호1가동,0.402,0.282,178,401,-223,0.000
57,성동구,금호4가동,0.389,0.272,201,405,-204,0.000
62,성동구,성수2가3동,0.385,0.269,208,407,-199,0.000
56,성동구,금호2.3가동,0.376,0.263,220,409,-189,0.000
268,구로구,고척2동,0.437,0.405,120,286,-166,0.330
295,영등포구,신길1동,0.475,0.431,77,242,-165,0.329


---
## 7. 등급 변동 분석 (grade 5 진입/이탈)

In [7]:
# IVI_grade는 string 'A/B/C/D/E' 형식 → 숫자로 변환
grade_map = {'A': 5, 'B': 4, 'C': 3, 'D': 2, 'E': 1}
df['IVI_grade_num'] = df['IVI_grade'].map(grade_map)

entered_5 = df[(df['IVI_v2_grade'] == 5) & (df['IVI_grade_num'] != 5)]
exited_5  = df[(df['IVI_v2_grade'] != 5) & (df['IVI_grade_num'] == 5)]

print(f'IVI v1 grade 5 (최고위험) 동: {(df["IVI_grade_num"]==5).sum()}개')
print(f'IVI v2 grade 5 (최고위험) 동: {(df["IVI_v2_grade"]==5).sum()}개')
print()
print(f'신규 진입 (v1 비최고위험 → v2 최고위험): {len(entered_5)}개')
if len(entered_5) > 0:
    display(entered_5[['gu','dong','IVI','IVI_v2','care_shortage_score']].round(3))
print()
print(f'이탈 (v1 최고위험 → v2 비최고위험): {len(exited_5)}개')
if len(exited_5) > 0:
    display(exited_5[['gu','dong','IVI','IVI_v2','care_shortage_score']].round(3))

IVI v1 grade 5 (최고위험) 동: 85개
IVI v2 grade 5 (최고위험) 동: 85개

신규 진입 (v1 비최고위험 → v2 최고위험): 27개


,gu,dong,IVI,IVI_v2,care_shortage_score
69,광진구,중곡3동,0.441,0.544,0.783
97,중랑구,면목본동,0.458,0.558,0.791
98,중랑구,면목7동,0.444,0.548,0.791
105,중랑구,묵2동,0.450,0.552,0.791
134,강북구,삼각산동,0.402,0.560,0.930
176,은평구,녹번동,0.457,0.552,0.772
177,은평구,불광1동,0.462,0.555,0.772
182,은평구,대조동,0.461,0.554,0.772
189,은평구,증산동,0.455,0.550,0.772
278,금천구,독산2동,0.446,0.549,0.789



이탈 (v1 최고위험 → v2 비최고위험): 27개


,gu,dong,IVI,IVI_v2,care_shortage_score
7,종로구,가회동,0.494,0.526,0.601
13,종로구,창신2동,0.510,0.537,0.601
15,종로구,숭인1동,0.499,0.530,0.601
16,종로구,숭인2동,0.494,0.526,0.601
42,용산구,이촌2동,0.485,0.509,0.564
47,용산구,보광동,0.482,0.507,0.564
80,동대문구,용신동,0.474,0.519,0.625
82,동대문구,전농1동,0.497,0.535,0.625
119,성북구,정릉3동,0.499,0.508,0.531
127,성북구,장위2동,0.516,0.521,0.531


---
## 8. 개포2동 검증

CCI triple_high 7개 동 중 강남구 개포2동이 포함된 배경 확인

In [8]:
# triple_high 7개 동의 IVI 구성 분해
triple_high_dongs = [
    ('강남구', '개포2동'),
    ('노원구', '상계3동'),
    ('강북구', '미아2동'),
    ('노원구', '중계2동'),
    ('중랑구', '신내2동'),
    ('노원구', '상계2동'),
    ('광진구', '군자동'),
]
th_df = pd.DataFrame(triple_high_dongs, columns=['gu', 'dong'])
th_detail = df.merge(th_df, on=['gu', 'dong'])

cols = ['gu', 'dong', 'elderly_ratio', 'elderly_alone_ratio',
        'elderly_ratio_score', 'elderly_alone_score', 'care_shortage_score',
        'IVI', 'IVI_v2', 'IVI_v2_rank']
print('=== triple_high 7개 동 — IVI v2 성분 분해 ===')
display(th_detail[cols].round(3))

print()
print('=== 강남구 전체 행정동 IVI v2 비교 ===')
gangnam = df[df['gu'] == '강남구'][['dong', 'elderly_ratio', 'elderly_alone_ratio',
                                    'care_shortage_score', 'IVI', 'IVI_v2', 'IVI_v2_rank']]
display(gangnam.sort_values('IVI_v2_rank').round(3))

print()
print(f'강남구 care_shortage_score: {care.loc[care["gu"]=="강남구", "care_shortage_score"].values[0]:.3f}')
print(f'강남구 경로당_per_1000eld: {care.loc[care["gu"]=="강남구", "경로당_per_1000eld"].values[0]:.2f}')

=== triple_high 7개 동 — IVI v2 성분 분해 ===


,gu,dong,elderly_ratio,elderly_alone_ratio,elderly_ratio_score,elderly_alone_score,care_shortage_score,IVI,IVI_v2,IVI_v2_rank
0,광진구,군자동,0.140,0.283,0.265,0.445,0.783,0.355,0.483,159
1,중랑구,신내2동,0.200,0.277,0.519,0.435,0.791,0.477,0.571,51
2,노원구,상계2동,0.168,0.252,0.385,0.396,0.384,0.391,0.389,304
3,강남구,개포2동,0.130,0.140,0.223,0.220,0.559,0.222,0.323,375



=== 강남구 전체 행정동 IVI v2 비교 ===


,dong,elderly_ratio,elderly_alone_ratio,care_shortage_score,IVI,IVI_v2,IVI_v2_rank
379,수서동,0.312,0.410,0.559,0.822,0.743,6
372,개포1동,0.250,0.220,0.559,0.539,0.545,80
377,일원1동,0.217,0.296,0.559,0.529,0.538,88
378,일원2동,0.194,0.302,0.559,0.486,0.508,131
361,압구정동,0.196,0.159,0.559,0.375,0.430,247
358,신사동,0.187,0.171,0.559,0.367,0.424,254
375,세곡동,0.151,0.193,0.559,0.308,0.383,312
360,논현2동,0.148,0.195,0.559,0.302,0.379,319
363,삼성1동,0.164,0.151,0.559,0.302,0.379,320
359,논현1동,0.131,0.236,0.559,0.298,0.376,324



강남구 care_shortage_score: 0.559
강남구 경로당_per_1000eld: 2.39


---
## 9. 최종 저장

In [9]:
SAVE_COLS = [
    'year', 'gu', 'dong', 'dong_key',
    'total_population', 'elderly_population', 'elderly_alone',
    'elderly_ratio', 'elderly_alone_ratio',
    'elderly_ratio_score', 'elderly_alone_score',
    'care_shortage_score', '경로당_per_1000eld',
    'IVI', 'IVI_rank', 'IVI_grade',         # v1 보존
    'IVI_v2', 'IVI_v2_rank', 'IVI_v2_grade',
]
df_out = df[SAVE_COLS].copy()
df_out.to_csv(OUT_IVI_V2, index=False, encoding='utf-8-sig')
print(f'저장 완료: {OUT_IVI_V2}')
print(f'행 수: {len(df_out)}, 열 수: {len(df_out.columns)}')

# 검증
print()
print('=== 최종 검증 ===')
v = pd.read_csv(OUT_IVI_V2, encoding='utf-8-sig')
print(f'결측치 (IVI_v2): {v["IVI_v2"].isna().sum()}건')
print(f'IVI_v2 0~1 여부: {bool(v["IVI_v2"].between(0, 1).all())}')
print(f'IVI_v2_grade 1~5 여부: {bool(v["IVI_v2_grade"].between(1, 5).all())}')
print(f'행 수 (기대: 426): {len(v)}')
print(f'IVI_v2 평균: {v["IVI_v2"].mean():.4f}, 표준편차: {v["IVI_v2"].std():.4f}')
print('✓ 검증 완료')

저장 완료: c:\Tsum2026\T_SUM2026\data\processed\ivi_social_isolation_2021_v2.csv
행 수: 426, 열 수: 19

=== 최종 검증 ===
결측치 (IVI_v2): 0건
IVI_v2 0~1 여부: True
IVI_v2_grade 1~5 여부: True
행 수 (기대: 426): 426
IVI_v2 평균: 0.4522, 표준편차: 0.1101
✓ 검증 완료
